# Colab guide: open a CSV, plot the data, apply FFT, and visualize the result

This notebook shows a simple workflow:
1. Upload a CSV file
2. Read it with pandas
3. Inspect the columns
4. Plot the signal in the time domain
5. Apply FFT
6. Plot the frequency spectrum
7. Find the dominant frequency

Expected CSV structure:
- one time column, for example `time_s`
- one signal column, for example `value`

If your column names are different, change them in the later cells.

Library documentation:
- Colab overview: https://colab.google/
- pandas `read_csv()`: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html
- pandas `DataFrame.head()`: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html
- Matplotlib pyplot tutorial: https://matplotlib.org/stable/tutorials/pyplot.html
- Matplotlib pyplot reference: https://matplotlib.org/stable/api/pyplot_summary.html
- NumPy FFT overview: https://numpy.org/doc/stable/reference/routines.fft.html
- NumPy `fft()`: https://numpy.org/doc/stable/reference/generated/numpy.fft.fft.html
- NumPy `fftfreq()`: https://numpy.org/doc/stable/reference/generated/numpy.fft.fftfreq.html
- SciPy FFT tutorial: https://docs.scipy.org/doc/scipy/tutorial/fft.html

## Step 1: Import the libraries

This cell imports the libraries used in the notebook.

- `pandas` reads and handles tabular data
- `matplotlib` draws plots
- `numpy` handles arrays and FFT
- `files` lets you upload a file from your computer

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

## Step 2: Upload the CSV file

Run the next cell and choose your CSV file from your computer.

In [ ]:
uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file:", filename)

## Step 3: Read the CSV file

This reads the CSV into a pandas DataFrame and shows the first rows.

In [ ]:
df = pd.read_csv(filename)
df.head()

## Step 4: Check the column names

This helps you see what columns are available and how many rows and columns the file contains.

In [ ]:
print(df.columns)
print(df.shape)

## Step 5: Select the time and signal columns and clean the data

This example assumes the CSV has:
- `time_s` for time in seconds
- `value` for the signal

If your CSV uses different names, replace them here.

This step also:
- converts the values to numeric
- removes invalid rows
- sorts the rows by time
- creates NumPy arrays for plotting and FFT

In [ ]:
time_col = "time_s"
value_col = "value"

data = df[[time_col, value_col]].copy()

data[time_col] = pd.to_numeric(data[time_col], errors="coerce")
data[value_col] = pd.to_numeric(data[value_col], errors="coerce")

data = data.dropna().sort_values(time_col)

t = data[time_col].to_numpy()
x = data[value_col].to_numpy()

print("Number of valid samples:", len(data))
data.head()

## Step 6: Plot the original signal

This shows the signal in the time domain. It helps you see trends, cycles, spikes, or missing-value problems.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, x)
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Signal in time domain")
plt.grid(True)
plt.show()

## Step 7: Calculate the sampling interval

FFT needs the sample spacing so frequencies can be mapped correctly.

If your time values are in seconds, the resulting frequency unit is Hz.

In [ ]:
dt = np.mean(np.diff(t))
fs = 1 / dt

print("Average sample spacing dt:", dt)
print("Sampling frequency fs:", fs)

## Step 8: Apply FFT

This step:
- removes the mean value from the signal
- computes the FFT
- computes the matching frequency bins
- keeps only the non-negative frequencies for plotting

In [ ]:
x_centered = x - np.mean(x)

n = len(x_centered)
fft_values = np.fft.fft(x_centered)
freqs = np.fft.fftfreq(n, d=dt)

positive = freqs >= 0
freqs_pos = freqs[positive]
amplitude_pos = (2.0 / n) * np.abs(fft_values[positive])

## Step 9: Plot the FFT magnitude spectrum

This plot shows which frequencies are strongest in the signal.

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(freqs_pos, amplitude_pos)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT magnitude spectrum")
plt.grid(True)
plt.show()

## Step 10: Find the dominant frequency

This finds the strongest non-zero frequency component and converts it to a period.

In [ ]:
peak_idx = np.argmax(amplitude_pos[1:]) + 1

dominant_freq = freqs_pos[peak_idx]
dominant_amp = amplitude_pos[peak_idx]
dominant_period = 1 / dominant_freq if dominant_freq != 0 else np.inf

print("Dominant frequency:", dominant_freq, "Hz")
print("Dominant amplitude:", dominant_amp)
print("Dominant period:", dominant_period, "s")

## Step 11: Optional zoom for low frequencies

If the spectrum is crowded, you can plot only a smaller frequency range.

In [ ]:
max_freq_to_show = 10  # Hz

mask = freqs_pos <= max_freq_to_show

plt.figure(figsize=(12, 4))
plt.plot(freqs_pos[mask], amplitude_pos[mask])
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT magnitude spectrum (zoomed)")
plt.grid(True)
plt.show()

## Full minimal version in one block

This is the same workflow in one code cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))

df = pd.read_csv(filename)

time_col = "time_s"
value_col = "value"

data = df[[time_col, value_col]].copy()
data[time_col] = pd.to_numeric(data[time_col], errors="coerce")
data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
data = data.dropna().sort_values(time_col)

t = data[time_col].to_numpy()
x = data[value_col].to_numpy()

plt.figure(figsize=(12, 4))
plt.plot(t, x)
plt.xlabel("Time (s)")
plt.ylabel("Signal")
plt.title("Signal in time domain")
plt.grid(True)
plt.show()

dt = np.mean(np.diff(t))
x_centered = x - np.mean(x)

n = len(x_centered)
fft_values = np.fft.fft(x_centered)
freqs = np.fft.fftfreq(n, d=dt)

positive = freqs >= 0
freqs_pos = freqs[positive]
amplitude_pos = (2.0 / n) * np.abs(fft_values[positive])

plt.figure(figsize=(12, 4))
plt.plot(freqs_pos, amplitude_pos)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("FFT magnitude spectrum")
plt.grid(True)
plt.show()

peak_idx = np.argmax(amplitude_pos[1:]) + 1
dominant_freq = freqs_pos[peak_idx]
dominant_period = 1 / dominant_freq if dominant_freq != 0 else np.inf

print("Dominant frequency:", dominant_freq, "Hz")
print("Dominant period:", dominant_period, "s")

## Common issues

### 1. `KeyError: 'time_s'`
Your CSV does not use that column name.

Run:
```python
print(df.columns)
```

Then replace:
```python
time_col = "your_actual_time_column"
value_col = "your_actual_signal_column"
```

### 2. FFT plot looks wrong
Common reasons:
- time values are not evenly spaced
- there are missing rows
- the wrong signal column was selected
- the signal has a strong trend or offset

### 3. Frequency unit is not Hz
`np.fft.fftfreq()` uses the unit of your time spacing.
- seconds -> Hz
- minutes -> cycles per minute